In [1]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))
import data_lib.stocks.stocks_config as sc

df = pd.read_hdf("../reports/test_scored.h5", "df")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (17578, 107)
Columns: ['mobile', 'latitude', 'longitude', 'nmbr_partners_x', 'decision_time', 'installed_decision', 'installed_time', 'hard_density', 'density_regime', 'local_anisotropy', 'local_density', 'hull_area', 'linearity_score', 'spread_m', 'dense_score', 'gully_score', 'sparse_score', 'geometry_regime', 'nmbr_overlap_clusters', 'mean_dist_to_edge_m', 'near_edge_instances', 'mean_dist_to_center_m', 'total_area_boundaries', 'n_hexes', 'nmbr_partners_y', 'worst_depth_score', 'any_near_edge', 'is_solo_cluster', 'nearest_boundary_dist_m', 'nmbr_boundaries_within_100m', 'n_overlapping_partners', 'contested_area_km2', 'contested_radius_m', 'contested_centroid_lat', 'contested_centroid_lon', 'contested_installs', 'contested_total', 'contested_se', 'contested_field', 'partner_id', 'poly_id', 'parent_overlap', 'parent_hex_dist_to_foreign_m', 'parent_hex_dist_to_own_m', 'parent_rank', 'best_size', 'dist_to_boundary_edge_point_hex', 'dist_to_cluster_center_point_hex', 'near_edge_po

In [2]:
# ── CONFIGURE: swap these if column names differ ──
SPATIAL_COL  = "composite_score"    # spatial composite (field, contested, log_ev, parent_se, color)
PARTNER_COL  = "operational_score"  # partner ops score
TARGET_COL   = "installed_decision"
COUNT_COL    = "mobile"

# ── response time: minutes → days ──
if "median_response_min" in df.columns:
    df["median_response_days"] = df["median_response_min"] / 1440
    df["mean_response_days"]   = df["mean_response_min"]   / 1440
    print(f"Response time converted: median range [{df['median_response_days'].min():.2f}, {df['median_response_days'].max():.2f}] days")

In [3]:
# ── Three composite variants ──
spatial  = df[SPATIAL_COL]
partner  = df[PARTNER_COL]

df["score_multiplicative"] = partner * spatial
df["score_mean"]           = (partner + spatial) / 2
df["score_spatial_heavy"]  = (partner + spatial * 3) / 4

SCORE_VARIANTS = {
    "score_multiplicative": "partner × spatial",
    "score_mean":           "(partner + spatial) / 2",
    "score_spatial_heavy":  "(partner + 3·spatial) / 4",
}

# sanity check
for col, label in SCORE_VARIANTS.items():
    print(f"{label:30s}  min={df[col].min():.4f}  max={df[col].max():.4f}  median={df[col].median():.4f}")

partner × spatial               min=0.0000  max=0.7856  median=0.1660
(partner + spatial) / 2         min=0.0000  max=0.8928  median=0.4381
(partner + 3·spatial) / 4       min=0.0000  max=0.8392  median=0.3588


In [4]:
def decile_table(df, score_col, target_col, count_col):
    col = f"{score_col}_decile"
    df[col] = pd.qcut(df[score_col], q=10, labels=False, duplicates="drop") + 1
    tbl = df.groupby(col).agg(
        n=(count_col, "count"),
        installs=(target_col, "sum"),
    ).assign(se=lambda x: x.installs / x.n)
    gap = tbl["se"].iloc[-1] - tbl["se"].iloc[0]
    monotonic = all(tbl["se"].diff().dropna() >= 0)
    return tbl, gap, monotonic

results = {}
for col, label in SCORE_VARIANTS.items():
    tbl, gap, mono = decile_table(df, col, TARGET_COL, COUNT_COL)
    results[col] = {"table": tbl, "gap": gap, "monotonic": mono, "label": label}
    print(f"\n{'='*50}")
    print(f"{label}  |  gap={gap:.4f}  |  monotonic={mono}")
    print(f"{'='*50}")
    print(tbl)


partner × spatial  |  gap=0.0535  |  monotonic=False
                                n  installs        se
score_multiplicative_decile                          
1                            1758       929  0.528441
2                            1758       927  0.527304
3                            1758       988  0.562002
4                            1757       978  0.556631
5                            1758      1054  0.599545
6                            1758      1105  0.628555
7                            1757      1136  0.646557
8                            1758      1148  0.653015
9                            1758      1189  0.676337
10                           1758      1023  0.581911

(partner + spatial) / 2  |  gap=0.0307  |  monotonic=False
                      n  installs        se
score_mean_decile                          
1                  1758       934  0.531286
2                  1758       932  0.530148
3                  1758      1042  0.592719
4                 

In [5]:
# ── Side-by-side comparison ──
summary = pd.DataFrame({
    label: {"decile_gap": r["gap"], "monotonic": r["monotonic"],
            "D1_se": r["table"]["se"].iloc[0], "D10_se": r["table"]["se"].iloc[-1]}
    for label, r in zip(
        [r["label"] for r in results.values()],
        results.values()
    )
}).T
summary = summary.sort_values("decile_gap", ascending=False)
print(summary.to_string())

                          decile_gap monotonic     D1_se    D10_se
(partner + 3·spatial) / 4   0.073948     False  0.531286  0.605233
partner × spatial            0.05347     False  0.528441  0.581911
(partner + spatial) / 2     0.030717     False  0.531286  0.562002


In [6]:
# ── Baseline: original composite_score ──
tbl_base, gap_base, mono_base = decile_table(df, SPATIAL_COL, TARGET_COL, COUNT_COL)
print(f"Baseline ({SPATIAL_COL}):  gap={gap_base:.4f}  monotonic={mono_base}")
print(tbl_base)

print(f"\n── vs new variants ──")
for col, r in results.items():
    delta = r['gap'] - gap_base
    print(f"  {r['label']:30s}  gap={r['gap']:.4f}  delta={delta:+.4f}")

Baseline (composite_score):  gap=0.1126  monotonic=False
                           n  installs        se
composite_score_decile                          
1                       1758       926  0.526735
2                       1758       916  0.521047
3                       1758       951  0.540956
4                       1757       986  0.561184
5                       1758      1016  0.577929
6                       1758      1082  0.615472
7                       1757      1117  0.635743
8                       1758      1115  0.634243
9                       1758      1244  0.707622
10                      1758      1124  0.639363

── vs new variants ──
  partner × spatial               gap=0.0535  delta=-0.0592
  (partner + spatial) / 2         gap=0.0307  delta=-0.0819
  (partner + 3·spatial) / 4       gap=0.0739  delta=-0.0387


In [7]:
# ── Per parent_color_super breakdown for each variant ──
if "parent_color_super" in df.columns:
    for col, r in results.items():
        print(f"\n── {r['label']} by color ──")
        print(df.groupby("parent_color_super").agg(
            n=(COUNT_COL, "count"),
            se=(TARGET_COL, "mean"),
            score_mean=(col, "mean"),
            score_std=(col, "std"),
        ).round(4))


── partner × spatial by color ──
                       n      se  score_mean  score_std
parent_color_super                                     
crimson             4963  0.5194      0.0975     0.0591
lightgreen          3179  0.6590      0.2421     0.0732
orange              8719  0.6181      0.1761     0.0694

── (partner + spatial) / 2 by color ──
                       n      se  score_mean  score_std
parent_color_super                                     
crimson             4963  0.5194      0.3528     0.1248
lightgreen          3179  0.6590      0.4925     0.0877
orange              8719  0.6181      0.4324     0.1044

── (partner + 3·spatial) / 4 by color ──
                       n      se  score_mean  score_std
parent_color_super                                     
crimson             4963  0.5194      0.2568     0.1012
lightgreen          3179  0.6590      0.4505     0.0863
orange              8719  0.6181      0.3633     0.0991


In [8]:
# ── Confidence tier × variant ──
for col, r in results.items():
    print(f"\n── {r['label']} by confidence_tier ──")
    tiers = np.select(
        [df[col] >= sc.R_TIER_MOD, df[col] >= sc.R_TIER_LOW, df[col] >= sc.R_TIER_DECLINE],
        ["HIGH", "MOD", "LOW"],
        default="DECLINE",
    )
    for tier in ["HIGH", "MOD", "LOW", "DECLINE"]:
        sub = df[tiers == tier]
        if len(sub):
            print(f"  {tier:8s}  n={len(sub):5d}  SE={sub[TARGET_COL].mean():.4f}  score_mean={sub[col].mean():.4f}")


── partner × spatial by confidence_tier ──
  HIGH      n=  231  SE=0.4199  score_mean=0.4329
  MOD       n= 5645  SE=0.6478  score_mean=0.2433
  LOW       n= 7590  SE=0.6000  score_mean=0.1532
  DECLINE   n= 4112  SE=0.5275  score_mean=0.0375

── (partner + spatial) / 2 by confidence_tier ──
  HIGH      n=14759  SE=0.6079  score_mean=0.4556
  MOD       n=  796  SE=0.5289  score_mean=0.3264
  LOW       n= 1433  SE=0.5429  score_mean=0.1228
  DECLINE   n=  590  SE=0.5186  score_mean=0.0000

── (partner + 3·spatial) / 4 by confidence_tier ──
  HIGH      n= 9309  SE=0.6385  score_mean=0.4291
  MOD       n= 5860  SE=0.5578  score_mean=0.2954
  LOW       n=  910  SE=0.4945  score_mean=0.1468
  DECLINE   n= 1499  SE=0.5430  score_mean=0.0371


In [9]:
# ── ML-optimized tier boundaries ──
# Objective: maximize (high_se - mid_se) + 3*(mid_se - low_se)
# Constraint: each tier has ≥ 1000 values
# Two thresholds → three tiers: LOW, MID, HIGH

from scipy.optimize import differential_evolution

def optimize_tiers(df, score_col, target_col, count_col, min_per_tier=1000):
    # pre-sort as plain Python lists
    data = sorted(zip(
        df[score_col].tolist(),
        df[target_col].tolist(),
    ))
    n = len(data)
    s_arr = [d[0] for d in data]
    t_arr = [d[1] for d in data]
    
    # prefix sums for O(1) range queries
    t_cum = [0.0] * (n + 1)
    for i in range(n):
        t_cum[i+1] = t_cum[i] + float(t_arr[i])
    
    def se_range(lo_idx, hi_idx):
        """SE = installs / n_rows  (same as decile_table)"""
        t = t_cum[hi_idx] - t_cum[lo_idx]
        c = hi_idx - lo_idx      
        return t / c if c > 0 else 0.0
    
    lo = np.percentile(s_arr, 1)
    hi = np.percentile(s_arr, 99)
    
    def neg_objective(params):
        t_low, t_high = sorted(params)
        i_low = int(np.searchsorted(s_arr, t_low))
        i_high = int(np.searchsorted(s_arr, t_high))
        
        if min(i_low, i_high - i_low, n - i_high) < min_per_tier:
            return 0.0
        
        se_l = se_range(0, i_low)
        se_m = se_range(i_low, i_high)
        se_h = se_range(i_high, n)
        
        return -((se_h - se_m) + 3.0 * (se_m - se_l) + 2.0 * (se_h - se_l))
    
    result = differential_evolution(
        neg_objective,
        bounds=[(lo, hi), (lo, hi)],
        seed=42, maxiter=500, tol=1e-8, polish=True,
    )
    t_low, t_high = sorted(result.x)
    return t_low, t_high, -result.fun


In [10]:
import numpy as np
import pandas as pd

# ensure numeric
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

for col, r in results.items():
    df[col] = pd.to_numeric(df[col], errors="coerce")

    t_low, t_high, obj = optimize_tiers(df, col, TARGET_COL, COUNT_COL)

    tiers = np.select(
        [df[col] >= t_high, df[col] >= t_low],
        ["HIGH", "MID"],
        default="LOW",
    )

    print(f"\n── {r['label']} ──")
    print(f"thresholds: LOW < {t_low:.4f} < MID < {t_high:.4f} < HIGH")
    print(f"objective: {obj:.4f}")

    for tier in ["HIGH", "MID", "LOW"]:
        sub = df[tiers == tier]
        se = sub[TARGET_COL].sum() / len(sub) if len(sub) else 0.0

        print(
            f"{tier:6s} n={len(sub):5d} "
            f"SE={se:.4f} score=[{sub[col].min():.4f}, {sub[col].max():.4f}]"
        )



── partner × spatial ──
thresholds: LOW < 0.0607 < MID < 0.2105 < HIGH
objective: 0.5148
HIGH   n= 4935 SE=0.6385 score=[0.2105, 0.7856]
MID    n= 9926 SE=0.5961 score=[0.0607, 0.2105]
LOW    n= 2717 SE=0.5186 score=[0.0000, 0.0607]

── (partner + spatial) / 2 ──
thresholds: LOW < 0.3687 < MID < 0.4656 < HIGH
objective: 0.4293
HIGH   n= 5938 SE=0.6243 score=[0.4656, 0.8928]
MID    n= 8121 SE=0.6039 score=[0.3688, 0.4656]
LOW    n= 3519 SE=0.5303 score=[0.0000, 0.3686]

── (partner + 3·spatial) / 4 ──
thresholds: LOW < 0.2300 < MID < 0.4273 < HIGH
objective: 0.5417
HIGH   n= 4277 SE=0.6535 score=[0.4273, 0.8392]
MID    n=10530 SE=0.5925 score=[0.2300, 0.4273]
LOW    n= 2771 SE=0.5208 score=[0.0000, 0.2300]


In [11]:
# ── Compare: config thresholds vs optimized ──
best_col = max(results, key=lambda c: optimize_tiers(df, c, TARGET_COL, COUNT_COL)[2])
t_low, t_high, obj = optimize_tiers(df, best_col, TARGET_COL, COUNT_COL)
label = results[best_col]["label"]

print(f"Best variant: {label}")
print(f"Optimized thresholds: {t_low:.4f}, {t_high:.4f}  (obj={obj:.4f})")
print(f"\nConfig thresholds:    DECLINE={sc.R_TIER_DECLINE}, LOW={sc.R_TIER_LOW}, MOD={sc.R_TIER_MOD}")

# side-by-side SE comparison
print(f"\n{'':30s} {'n':>6s} {'SE':>8s}")
for label_name, mask in [
    ("Optimized HIGH", df[best_col] >= t_high),
    ("Optimized MID",  (df[best_col] >= t_low) & (df[best_col] < t_high)),
    ("Optimized LOW",  df[best_col] < t_low),
]:
    sub = df[mask]
    se = sub[TARGET_COL].sum() / len(sub) if len(sub) else 0.0
    print(f"  {label_name:28s} {len(sub):6d} {se:8.4f}")


Best variant: (partner + 3·spatial) / 4
Optimized thresholds: 0.2300, 0.4273  (obj=0.5417)

Config thresholds:    DECLINE=0.1, LOW=0.2, MOD=0.35

                                    n       SE
  Optimized HIGH                 4277   0.6535
  Optimized MID                 10530   0.5925
  Optimized LOW                  2771   0.5208


In [14]:
from scipy.optimize import differential_evolution

def optimize_tiered_score(df, spatial_col, partner_col, target_col, min_per_tier=1000):
    spatial = df[spatial_col].values.astype(float)
    partner = df[partner_col].values.astype(float)
    target  = df[target_col].values.astype(float)
    n = len(df)

    def objective(params):
        w = params[0]                        # alpha / (alpha + beta)
        t1, t2, t3 = sorted(params[1:4])     # 3 thresholds → 4 tiers

        score = w * spatial + (1 - w) * partner

        decline = target[score < t1]
        low     = target[(score >= t1) & (score < t2)]
        mid     = target[(score >= t2) & (score < t3)]
        high    = target[score >= t3]

        if min(len(decline), len(low), len(mid), len(high)) < min_per_tier:
            return 0.0

        se_d = decline.mean()
        se_l = low.mean()
        se_m = mid.mean()
        se_h = high.mean()
        if not (se_h > se_m > se_l > se_d):
            return 0.0

        return -(
            (se_h - se_m)
            + 2.0 * (se_m - se_l)
            + 3.0 * (se_l - se_d)
            + 2.0 * (se_h - se_d)
        )

    # bounds: w in [0,1], thresholds in score range
    s_lo = min(spatial.min(), partner.min())
    s_hi = max(spatial.max(), partner.max())

    
    result = differential_evolution(
        objective,
        bounds=[(0.2, 0.8), (s_lo, s_hi), (s_lo, s_hi), (s_lo, s_hi)],
        seed=42, maxiter=1000, tol=1e-9, polish=True,
    )
    

    w = result.x[0]
    t1, t2, t3 = sorted(result.x[1:4])
    alpha, beta = w, 1 - w

    return alpha, beta, t1, t2, t3, -result.fun


alpha, beta, t1, t2, t3, obj = optimize_tiered_score(
    df, SPATIAL_COL, PARTNER_COL, TARGET_COL
)

print(f"alpha={alpha:.4f}  beta={beta:.4f}  (spatial weight={alpha/(alpha+beta):.2%})")
print(f"thresholds: {t1:.4f} / {t2:.4f} / {t3:.4f}")
print(f"objective: {obj:.4f}\n")

df["blended_score"] = (alpha * df[SPATIAL_COL] + beta * df[PARTNER_COL]) / (alpha + beta)

print(f"{'TIER':>8s} {'n':>6s} {'SE':>8s} {'score_range':>20s}")
for tier, lo, hi in [
    ("DECLINE", None, t1), ("LOW", t1, t2), ("MID", t2, t3), ("HIGH", t3, None)
]:
    mask = pd.Series(True, index=df.index)
    if lo is not None: mask &= df["blended_score"] >= lo
    if hi is not None: mask &= df["blended_score"] < hi
    sub = df[mask]
    se = sub[TARGET_COL].mean() if len(sub) else 0.0
    print(f"{tier:>8s} {len(sub):6d} {se:8.4f} [{sub['blended_score'].min():.4f}, {sub['blended_score'].max():.4f}]")

tbl, gap, mono = decile_table(df, "blended_score", TARGET_COL, COUNT_COL)
print(f"\nDecile gap={gap:.4f}  monotonic={mono}")
print(tbl)

alpha=0.7989  beta=0.2011  (spatial weight=79.89%)
thresholds: 0.2063 / 0.4112 / 0.4282
objective: 0.6054

    TIER      n       SE          score_range
 DECLINE   2777   0.5200 [0.0000, 0.2063]
     LOW  10246   0.5911 [0.2063, 0.4112]
     MID   1004   0.6534 [0.4112, 0.4282]
    HIGH   3551   0.6536 [0.4282, 0.8287]

Decile gap=0.0825  monotonic=False
                         n  installs        se
blended_score_decile                          
1                     1758       937  0.532992
2                     1758       917  0.521615
3                     1758       973  0.553470
4                     1757       968  0.550939
5                     1758      1052  0.598407
6                     1758      1082  0.615472
7                     1757      1119  0.636881
8                     1758      1134  0.645051
9                     1758      1213  0.689989
10                    1758      1082  0.615472
